In [ ]:
import torch
from transformers import AutoTokenizer
from transformers.models.gemma3 import Gemma3ForCausalLM
import os
import re
import spacy
from datasets import Dataset, DatasetDict, load_dataset
import json
from tqdm import tqdm

def fix_tags_with_replace(text):
    """
    Fixes the formatting of tags in the text using replace for each possible case.

    Args:
        text (str): The text generated by the model with malformed tags.

    Returns:
        str: Text with corrected tags.
    """
    # List of tags that need to be corrected
    tags = [
        "AGE", "PHONE", "FAX", "EMAIL", "URL", "IP_ADDRESS", "DATE", "IDNUM",
        "MEDICAL_RECORD", "DEVICE", "HEALTH_PLAN", "BIOID", "STREET", "CITY",
        "ZIP", "STATE", "COUNTRY", "LOCATION_OTHER", "ORGANIZATION", "HOSPITAL",
        "PATIENT", "DOCTOR", "USERNAME", "PROFESSION", "OTHER", "LOCATION"
    ]

    for tag in tags:
       # Fix spaces around the opening tag
        text = text.replace(f"< {tag} >", f"<{tag}>").replace(f"< {tag}>", f"<{tag}>").replace(f"<{tag} >", f"<{tag}>")
        # Fix spaces around the closing tag
        text = text.replace(f"</ {tag} >", f"</{tag}/>").replace(f"</ {tag}>", f"</{tag}/>").replace(f"</{tag} >", f"</{tag}/>")
        # Fix malformed closing tags with extra slashes
        text = text.replace(f"<{tag}/> ", f"</{tag}/>").replace(f"<{tag}/ >", f"</{tag}/>").replace(f"</{tag}/ >", f"</{tag}/>")
        # Remove spaces between tags and their inner content
        text = text.replace(f"<{tag}> ", f"<{tag}>").replace(f" </{tag}>", f"</{tag}/>").replace(f"</{tag}>", f"</{tag}/>")

    return text

def sliding_window(text, window_size=200, overlap=50):
    """
    Function that splits a text into sliding windows of size `window_size`
    with an overlap of `overlap`.

    Args:
        text (str): The text to be processed.
        window_size (int): The maximum number of tokens per window.
        overlap (int): The number of overlapping tokens between windows.

    Returns:
        List of str: List containing the text split into sliding windows.
    """
     # Load the SpaCy model for Portuguese (or another language, if needed)
    nlp = spacy.load('pt_core_news_sm')

    # Process the full text with SpaCy
    doc = nlp(text)

    # Extract the tokens
    tokens = [token.text for token in doc]

    # List to store the text windows
    windows = []
    
    # Sliding window implementation
    for i in range(0, len(tokens), window_size - overlap):
        # Capture a window of tokens
        window = tokens[i:i + window_size]
        windows.append(fix_tags_with_replace(" ".join(window)))

        # Stop if the text comes to a end
        if i + window_size >= len(tokens):
            break

    return windows


def extract_tagged_words(text):
    """
    Create labels
    """
    dict_categories = {"AGE": "AGE", "PHONE": "CONTACT", "FAX": "CONTACT", "EMAIL": "CONTACT", "URL": "CONTACT",
                   "IP_ADDRESS": "CONTACT", "DATE": "DATE", "IDNUM": "ID", "MEDICAL_RECORD": "ID", "DEVICE": "ID",
                   "HEALTH_PLAN": "ID", "BIOID": "ID", "STREET": "LOCATION", "CITY": "LOCATION", "ZIP": "LOCATION",
                   "STATE": "LOCATION", "COUNTRY": "LOCATION", "LOCATION_OTHER": "LOCATION", "ORGANIZATION": "LOCATION",
                   "HOSPITAL": "LOCATION", "PATIENT": "NAME", "DOCTOR": "NAME", "USERNAME": "NAME", "PROFESSION": "PROFESSION",
                   "OTHER": "OTHER", "LOCATION": "LOCATION"}

    pattern = r"<(.*?)>(.*?)</\1/>"  
    matches = re.finditer(pattern, text)

    result = []
    for match in matches:
        tag = match.group(1)
        word = match.group(2)
        first_position = match.start(2)  
        last_position = match.end(2)     
        category = dict_categories.get(tag, "UNKNOWN")  
        subcategory = tag

        result.append({
            "word": word,
            "category": category,
            "subcategory": subcategory,
            "first_position": first_position,
            "last_position": last_position
        })

    return result

def find_missing_words(predicted_words, labels):
    """
    Finds the words present in `labels` that are not in `predicted_words`.

    Args:
        predicted_words (list): List of predicted words.
        labels (list): List of correct words (labels).

    Returns:
        tuple: Number of missing words and a list of those words.
    """
    missing_words = [word for word in labels if word not in predicted_words]
    return len(missing_words), missing_words

def calculate_f1_score(tp, fp, fn, verbose= False):
    """
    Calculate F1-score, recall, and precision.

    Args:
        tp (int): True positives.
        fp (int): False positives.
        fn (int): False negatives.
        verbose (bool, optional): If True, prints metrics.

    Returns:
        tuple: (f1_score, recall, precision)
    """
    # Calculate precision (Total correct predictions divided by the number of predictions made)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    # Calculate recall (Total correct predictions divided by the number of words that should have been masked)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    # Calculate the F1 Score
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    if verbose:
        print('Recall:', recall)
        print('Precision:', precision)
        print('F1 Score:', f1_score, '\n')

    return f1_score, recall, precision

def eval(extractive_pred, dict_labels, verbose=False):
    """
    Evaluate entity extraction predictions against ground-truth labels.

    The function compares predicted entities with reference labels,
    computes precision, recall, and F1-score, and provides insights
    on correct predictions, incorrect categories, and missing words.

    Args:
        extractive_pred (dict): Dictionary with predictions (must include 'preds').
        dict_labels (dict): Ground-truth labels mapping words to entity categories.
        verbose (bool, optional): If True, prints detailed evaluation results.

    Returns:
        tuple: (f1, recall, precision)
    """
    TP, FP, FN = 0, 0, 0
    correct_predicted_words = []
    wrong_predicted_words = []
    predicted_words = []
    wrong_predicted_category = []
    for pred in extractive_pred['preds']:

        if pred['word'] in list(dict_labels.keys()):

            if pred['subcategory'] == dict_labels[pred['word']]:
                TP+=1
                correct_predicted_words.append((pred['word'], pred['subcategory']))
                predicted_words.append(pred['word'])
            else:
                FP+=1
                wrong_predicted_category.append((pred['word'], pred['subcategory']))
        else:          
            FP+=1
            wrong_predicted_words.append(pred['word'])

    # Calculate False Negatives
    FN, missing_words = find_missing_words(predicted_words, list(dict_labels.keys()))
    if verbose:
        print('Missing words:', missing_words)
        print('Correct Predicted words:', correct_predicted_words)
        print('Correct word but wrong category:', wrong_predicted_category)
        print('Wrong Predicted words:', wrong_predicted_words)
        print('Labels:', dict_labels)

    # Calculate F1 Score
    f1, recall, precision = calculate_f1_score(TP, FP, FN, verbose=verbose)

    return f1, recall, precision

def create_generative_format(text):
    """
    Convert a text with annotated tags into a generative format.

    This function replaces each annotated tag of the form word
    with a simplified generative format , removing the inner content.

    Args:
        text (str): Text containing tagged entities.

    Returns:
        str: Text with entities in generative format.
    """
    # Regex to capture opening tag and content
    pattern = r"<(.*?)>(.*?)\1/>"

    # Replacement function that keeps only the opening tag
    def replace_match(match):
        tag = match.group(1)
        return f"<{tag}>"

    # Replace all occurrences in the text
    replaced_text = re.sub(pattern, replace_match, text)
    return replaced_text


def run_prediction(input_text, max_length=1024):
    """
    Run model inference on input text using a sliding window.

    Args:
        input_text (str): The input text to be processed.
        max_length (int): Maximum output length.

    Returns:
        list: List of decoded model predictions.
    """
    list_output = []
    for window in input_text:
        inputs = tokenizer(window, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_length=max_length)
        try:
            list_output.append(tokenizer.decode(outputs[0]).split("target_text:")[1].split('<eos>')[0])
        except:
            try:
                list_output.append(tokenizer.decode(outputs[0]).split("target_text:")[1])
            except:
                list_output.append(tokenizer.decode(outputs[0]))
    
    return list_output

### Instantiate the model
GEMMA_PATH = "<PATH_TO_FINE_TUNED_MODEL" 
max_seq_length = 1024

os.environ["HF_TOKEN"] = "<HF_TOKEN>"

tokenizer = AutoTokenizer.from_pretrained(GEMMA_PATH, token=os.environ["HF_TOKEN"])

model = Gemma3ForCausalLM.from_pretrained(
    GEMMA_PATH,
    token=os.environ["HF_TOKEN"],
    dtype=torch.bfloat16,  
    device_map="auto",         
    attn_implementation="sdpa", 
    low_cpu_mem_usage=True       
)

In [ ]:
save_path = "<SAVE_PATH>"

# Download dataset
dataset_ = load_dataset("Venturus/AnonyMED-BR_FULL")
test_set = dataset_['test']

list_f1 = []
list_recall = []
list_precision = []
list_pred_gen = []
list_syn = []
list_id = []
list_preds = []
for test_sample in tqdm(test_set):
    # Create the sliding window to input on the NLP model
    clean_text = re.sub(r"<[^>]+>", "", test_sample['text'])
    windows_ = sliding_window((clean_text), window_size=150, overlap=0)
    
    # Format inputs
    windows = [f"input_text: {window}\ntarget_text:" for window in windows_]
    
    # Run batch inference on all windows and post-process data
    preds = run_prediction(windows, 1024)

    final_pred = ' '.join(preds)
    fixed_final_pred = fix_tags_with_replace(final_pred).replace(' , ',', ').replace('</PHONE/>-<PHONE>','-').replace(' </PHONE/>','</PHONE/>').replace(' )</PHONE/>',')</PHONE/>')#.replace(' )',')').replace('( ','(')

    ### Save prediction in a list
    list_preds.append(fixed_final_pred)

    ### Extractive Format Evaluation ###
    # Get extractive predictions
    tags = extract_tagged_words(fixed_final_pred)
    extractive_pred = {'id': test_sample['id'], 'preds': tags}

    # Get labels in the evaluation format
    dict_labels = {item['word']: item['subcategory'] for item in test_sample['labels']}

    # Run evaluation
    f1, recall, precision = eval(extractive_pred, dict_labels, verbose=False)

    # Save results for correct words and classes
    list_f1.append(f1)
    list_recall.append(recall)
    list_precision.append(precision)

    # Save if it is synthetic or not
    list_syn.append(test_sample['synthetic'])

    # Save example id
    list_id.append(test_sample['id'])

    ### Generative Format Evaluation ###
    list_pred_gen.append({'text': re.sub(r"<[^>]+>", "", test_sample['text']), 'masked_text':create_generative_format(test_sample['text']), 'prediction':create_generative_format(fixed_final_pred)})


### Create a dataframe containing the evaluations
save_df = pd.DataFrame()
save_df['id'] = list_id
save_df['Recall'] = list_recall
save_df['Precision'] = list_precision
save_df['F1'] = list_f1
save_df['synthetic'] = list_syn
save_df['Prediction'] = list_preds
save_df.to_csv('{}/Results.csv'.format(save_path), index = False)

avg_f1 = sum(list_f1) / len(list_f1) if list_f1 else 0
avg_recall = sum(list_recall) / len(list_recall) if list_recall else 0
avg_precision = sum(list_precision) / len(list_precision) if list_precision else 0

print('Recall:', avg_recall)
print('Precision:', avg_precision)
print('F1:', avg_f1)

## Save predictions on the generative format
with open('{}/t5_generative_predictions.json'.format(save_path), 'w', encoding='utf-8') as f:
      json.dump(list_pred_gen, f, ensure_ascii=False, indent=4)
     